<a href="https://colab.research.google.com/github/avenircc120-debug/newaivideo/blob/main/colab_serveur.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 RÉALITÉ TOTALE 2026 — Serveur API Gradio
**Exécution : CPU | Tunnel : Gradio public (.gradio.live)**

### Instructions :
1. Exécute **toutes les cellules dans l'ordre** (Runtime → Run all)
2. À la fin, copie le lien `https://xxxx.gradio.live`
3. Colle ce lien dans ton fichier HTML

In [ ]:
# ── CELLULE 1 : Installation des dépendances ──────────────
!pip install -q gradio diffusers transformers accelerate \
    opencv-python-headless Pillow torch torchvision \
    controlnet-aux clip-interrogator basicsr facexlib \
    realesrgan huggingface-hub safetensors einops omegaconf

print('✅ Dépendances installées')

In [ ]:
# ── CELLULE 2 : Chargement du token HuggingFace depuis les Secrets Colab ──
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

# Connexion à HuggingFace Hub
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

print('✅ HF_TOKEN chargé et connecté')

In [ ]:
# ── CELLULE 3 : Cloner le dépôt newaivideo ────────────────
import os

if not os.path.exists('/content/newaivideo'):
    !git clone https://github.com/avenircc120-debug/newaivideo /content/newaivideo
else:
    !git -C /content/newaivideo pull

os.chdir('/content/newaivideo')
print('✅ Dépôt prêt dans /content/newaivideo')

In [ ]:
# ── CELLULE 4 : Créer la structure Maître à la racine ─────
import sys
sys.path.insert(0, '/content/newaivideo')

from mount_bridge import demarrer_systeme

systeme = demarrer_systeme()
CHEMIN_VITESSE = systeme['chemins']['vitesse']
print(f'✅ Sortie vidéo → {CHEMIN_VITESSE}')

In [ ]:
# ── CELLULE 5 : Moteur de génération ─────────────────────
import sys, os, json, time
from pathlib import Path
from PIL import Image
import numpy as np

sys.path.insert(0, '/content/newaivideo')

from prompt_interpreter import interpret_user_order
from points_matrix import build_2000_points_matrix
from pipeline import RealityPipeline


def generer_video(photo_numpy, description: str):
    """
    Fonction appelée par Gradio.
    photo_numpy : image uploadée par l'utilisateur (numpy array RGB)
    description : texte décrivant la scène
    Retourne : (chemin_video, rapport_texte)
    """
    if photo_numpy is None:
        return None, '❌ Veuillez charger votre photo.'
    if not description or not description.strip():
        return None, '❌ Veuillez écrire une description.'

    debut = time.time()

    # 1. Convertir la photo
    face_image = Image.fromarray(photo_numpy.astype('uint8')).convert('RGB')

    # 2. Écrire la description à la racine
    desc_path = Path(systeme['chemins']['description'])
    desc_path.parent.mkdir(parents=True, exist_ok=True)
    desc_path.write_text(
        json.dumps({'prompt': description}, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

    # 3. Analyser le prompt → activer les bons outils et points
    ordre = interpret_user_order(description)

    # 4. Construire la matrice 2000 points
    matrix = build_2000_points_matrix(overrides=ordre['point_overrides'])

    # 5. Lancer le pipeline 14 outils
    runner = RealityPipeline(
        face_image      = face_image,
        prompt          = ordre['enriched_prompt'],
        tools_activated = ordre['tools_activated'],
        matrix          = matrix,
        output_dir      = CHEMIN_VITESSE,
        resolution      = (1920, 1080),  # 1080p sur CPU, 4K avec GPU A100
        num_frames      = 72,
        fps             = 24,
        use_gpu         = False,         # CPU — passe à True avec GPU Vertex AI
    )
    result = runner.run()

    duree = time.time() - debut
    video_path = Path(CHEMIN_VITESSE) / 'video_finale.mp4'

    rapport = (
        f'✅ Généré en {duree:.1f}s\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'Scènes     : {chr(10).join(ordre["scene_labels"]) or "Standard"}\n'
        f'Outils     : {len(result["tools_ran"])}/14 actifs\n'
        f'Points IA  : {result["final_params"].get("total_points_injected", 0)}/2000\n'
        f'Résolution : 1920×1080\n'
        f'Sortie     : {video_path}\n'
    )

    return str(video_path) if result['success'] else None, rapport


print('✅ Moteur de génération prêt')

In [ ]:
# ── CELLULE 6 : Interface Gradio + Tunnel public ──────────
import gradio as gr

interface = gr.Interface(
    fn          = generer_video,
    inputs      = [
        gr.Image(
            label  = '📸 Votre photo (visage)',
            type   = 'numpy',
            height = 300,
        ),
        gr.Textbox(
            label       = '✍️ Description de la scène',
            placeholder = 'Ex: Moi, marchant à la foire de Cotonou, au soleil couchant.',
            lines       = 4,
        ),
    ],
    outputs     = [
        gr.Video(label  = '🎬 Vidéo générée'),
        gr.Textbox(label= '📊 Rapport de génération', lines=8),
    ],
    title       = '🎬 Réalité Totale 2026 — La Grande Chose',
    description = (
        '**Pipeline 14 outils IA · 2000 points de contrôle · Rendu 4K Arri Alexa**\n'
        'Chargez votre photo, décrivez la scène, et la machine génère votre vidéo.'
    ),
    examples    = [
        [None, 'Moi, marchant à la foire de l\'indépendance de Cotonou, au soleil couchant.'],
        [None, 'Moi, montant dans une voiture zémidjan, il pleut beaucoup.'],
        [None, 'Moi, en moto à 120 km/h, le vent tape mon visage.'],
    ],
    allow_flagging = 'never',
)

# ⚡ Lance le serveur avec tunnel public Gradio
# Le lien https://xxxx.gradio.live s'affiche juste en dessous
interface.launch(
    share        = True,    # Génère le lien public .gradio.live
    server_port  = 7860,
    show_error   = True,
    quiet        = False,
)